<a href="https://colab.research.google.com/github/Kmpi17/AccesoDatosG/blob/main/TFG_KMPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

df_products = pd.read_csv('/content/drive/MyDrive/GreenWashing_Data/productos_espana.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_9659/2633681760.py:8: DtypeWarning: Columns (0,11,17,34,36,52,54,68,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df_products = pd.read_csv('/content/drive/MyDrive/GreenWashing_Data/productos_espana.csv')


# **ANALISIS PREVIO DEL DATASET**

Celda usada para ver en mas profundidad el contenido del dataset

In [ ]:
print("TODO LOS NOMBRE DE COLUMNAS ")
print(df_products.columns.tolist())
print("------------")
print(df_products.iloc[1900].to_string())

TODO LOS NOMBRE DE COLUMNAS 
['code', 'url', 'creator', 'created_t', 'created_datetime', 'last_modified_t', 'last_modified_datetime', 'last_modified_by', 'last_updated_t', 'last_updated_datetime', 'product_name', 'abbreviated_product_name', 'generic_name', 'quantity', 'packaging', 'packaging_tags', 'packaging_en', 'packaging_text', 'brands', 'brands_tags', 'brands_en', 'categories', 'categories_tags', 'categories_en', 'origins', 'origins_tags', 'origins_en', 'manufacturing_places', 'manufacturing_places_tags', 'labels', 'labels_tags', 'labels_en', 'emb_codes', 'emb_codes_tags', 'first_packaging_code_geo', 'cities', 'cities_tags', 'purchase_places', 'stores', 'countries', 'countries_tags', 'countries_en', 'ingredients_text', 'ingredients_tags', 'ingredients_analysis_tags', 'allergens', 'allergens_en', 'traces', 'traces_tags', 'traces_en', 'serving_size', 'serving_quantity', 'no_nutrition_data', 'additives_n', 'additives', 'additives_tags', 'additives_en', 'nutriscore_score', 'nutriscore

Celda usada para ver informacion generica sobre el dataset

In [ ]:
print("Numero de columnas y nombres")
print(df_products.columns)
print(f"Total de columnas {len(df_products.columns)}")
print("------------")
print(f"Totel de registros {len(df_products)}")
print("------------")
print(f"Valores nulos")
nulos=df_products.isnull().sum().sum()
print(nulos)
print("------------")
print("Valores duplicados")
duplicados=df_products.duplicated().sum()
print(duplicados)
print("------------")
print("Tipos de datos")
print(df_products.dtypes)
print("------------")


Numero de columnas y nombres
Index(['code', 'url', 'creator', 'created_t', 'created_datetime',
       'last_modified_t', 'last_modified_datetime', 'last_modified_by',
       'last_updated_t', 'last_updated_datetime',
       ...
       'water-hardness_100g', 'choline_100g', 'phylloquinone_100g',
       'beta-glucan_100g', 'inositol_100g', 'carnitine_100g', 'sulphate_100g',
       'nitrate_100g', 'acidity_100g', 'carbohydrates-total_100g'],
      dtype='object', length=209)
Total de columnas 209
------------
Totel de registros 352707
------------
Valores nulos
60038572
------------
Valores duplicados
0
------------
Tipos de datos
code                         object
url                          object
creator                      object
created_t                     int64
created_datetime             object
                             ...   
carnitine_100g              float64
sulphate_100g               float64
nitrate_100g                float64
acidity_100g                float64
carb

# **PROCESO DE LIMPIEZA**

Previamente vamos al dataset a hacer una copia en una variable para posteriormente comparar entre ellos todos los cambios

In [ ]:
df_products_copy=df_products.copy()

ELIMINACION DE FILAS VACIAS

In [ ]:
print(f"Totel de registros antes:{len(df_products_copy)}")
df_products_copy=df_products_copy.dropna(how="all")
print(f"Totel de registros despues:{len(df_products_copy)}")

Totel de registros antes:352707
Totel de registros despues:352707


LIMPIEZA DE COLUMNAS INECESARIAS

In [ ]:
print(f"Numero de columnas antes:{len(df_products_copy.columns)}")
columnas_utilizadas=['code', 'product_name', 'brands', 'categories_en', 'labels_tags','nutriscore_grade', 'energy-kcal_100g',
    'fat_100g', 'saturated-fat_100g', 'sugars_100g', 'proteins_100g', 'salt_100g','additives_n','additives_tags']

df_products_copy=df_products_copy[columnas_utilizadas]

print(f"Numero de columnas despues:{len(df_products_copy.columns)}")

Numero de columnas antes:209
Numero de columnas despues:14


**LIMPIEZA DE VALORES NULOS**

en este caso vamos a eliminar todos los registros en los cuales las calorias y los azucares sean nulos debido a que se va a priorizar la integridad del modelo en un futuro sobre el volumen de datos

In [ ]:
print(f"Totel de registros antes:{len(df_products_copy)}")
df_products_copy=df_products_copy.dropna(subset=['energy-kcal_100g', 'sugars_100g'])
print(f"Totel de registros despues:{len(df_products_copy)}")


Totel de registros antes:352707
Totel de registros despues:20904


code                                                           27016282
product_name                                           Crème de gruyère
brands                                                         Valblanc
categories_en         Dairies,Fermented foods,Fermented milk product...
labels_tags                                                         NaN
nutriscore_grade                                                      e
energy-kcal_100g                                                  272.0
fat_100g                                                           23.0
saturated-fat_100g                                                 15.0
sugars_100g                                                         6.6
proteins_100g                                                       9.4
salt_100g                                                          2.25
additives_n                                                         2.0
additives_tags                                          en:e450,

**EXPLICACION DE DISMINUCION**

como se puede ver la cantidad de registros ha disminuido de gran forma esta es debido a que por un lado se esta priorizando la calidad del modelo por tanto no se usaran registros con informacion incompleta ya que nuestro modelo no seria acertado, por otro lado la API utilizada es comunitaria por tanto hay muchos registros "basura" con informacion incompleta, por tanto al priorizarse la calidad frente a la cantidad da este resultado

In [ ]:
print(f"Totel de registros antes:{len(df_products_copy)}")
df_products_copy=df_products_copy[(df_products_copy['fat_100g'] +
                                   df_products_copy['sugars_100g'] +
                                   df_products_copy['proteins_100g']) <= 100]

df_products_copy=df_products_copy[df_products_copy['fat_100g'] >= df_products_copy['saturated-fat_100g']]
df_products_copy=df_products_copy.dropna(subset=['product_name','code','brands','categories_en'])


print(f"Totel de registros despues:{len(df_products_copy)}")

Totel de registros antes:19953
Totel de registros despues:17848
